# Hybrid Object-Pixel based Classification Assembly

## 1. Setup the path for the source code

In [1]:
#path setup
#since the notebook is in a subfolder, we need to add the src folder to the path
#The issue can be solved by installing the package in editable mode
from pathlib import Path
import sys
project_root = next(parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "src" / "ML_LC_Classifier").is_dir())
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
from ML_LC_Classifier import mosaic_rasters

## 1.1 Perform Mosaicking (If necessary)
Mosaic land cover probability layer and segments

In [ ]:
from ML_LC_Classifier import mosaic_rasters
prob_2025_path = r"C:\AFLICM_Test\hybrid_lc\2025_prob\*.tif"
prob_2020_path = r"C:\AFLICM_Test\hybrid_lc\2020_prob\*.tif"
mosaic_prob_2025 = mosaic_rasters(prob_2025_path, 
                                  output_path=r"C:\AFLICM_Test\hybrid_lc\TL_Prob_2025_mosaic.tif",
                                  method='first',
                                  nodata=0)
mosaic_prob_2020 = mosaic_rasters(prob_2020_path, 
                                  output_path=r"C:\AFLICM_Test\hybrid_lc\TL_Prob_2020_mosaic.tif",
                                  method='first',
                                  nodata=0)


In [ ]:
segment_path = r"C:\AFLICM_Test\hybrid_lc\SNIC_segments\*.tif"
segments_mosaic = mosaic_rasters(segment_path, 
                                 output_path=r"C:\AFLICM_Test\hybrid_lc\segments_mosaic_v1.tif",
                                 method="first",
                                 nodata = -1)

## 2. Execute Classification Assembly
### 2025 land cover

In [2]:
from ML_LC_Classifier.hybrid_assembly import process_tile
#Setup the input and output
lc_probability_path = r"C:\AFLICM_Test\hybrid_lc\Mosaic_LC_Prob_Sentinel\TL_Prob_2025_mosaic.tif"
#20m segments:
#segments = r"C:\AFLICM_Test\hybrid_lc\S2_SNIC_segments\TL_SNIC_CCDC_segments_S2_20m.tif"
segments = r"C:\AFLICM_Test\hybrid_lc\S2_SNIC_segments\segments_mosaic.tif"
output_dir = Path(r"C:\AFLICM_Test\hybrid_lc\hybrid_outputs\1_new-stage1_assembly\new_test_2025")
#Define custom class weights (Scale: 0.1 to 5.0)
#Higher = trust segment more (smoother); Lower = trust pixel more (sharper)
smoothing_param = {
    # Dense contiguous natural vegetation (Strong segment smoothing)
    'Primary_forest':        4.8,
    'Secondary_forest':      4.5,
    'Coastal_forest':        3.0, #adjusted due to over estimations (original value: 4.0)
    'Mangrove_forest':       3.0, #adjusted due to over estimations (original value: 4.0)
    
    # Plantations & Agroforestry (Balanced 50/50 object-pixel blend)
    'Teak_plantation':       1.2, #adjusted due to over estimations (original value: 2.5)
    'Eucalyptus_plantation': 1.5,
    'Coffe_agroforestry':    2.5,
    'Cocoa_agroforestry':    2.5,
    'Coconut_agroforestry':  2.0,
    'Mixed_garden':          2.0,
    
    # Open / Herbaceous cover
    'Grassland':             2.5,
    'Shrubland':             2.5,
    'Other_cropland':        2.0,
    'Paddy_field':           2.0,  #Preserves narrow smallholder rice terraces
    
    # Fragmented, dynamic & linear elements (Pixel spectral fidelity dominant)
    'Cleared_land':          1.5,
    'Built-up':              1.5,  # Preserves small hamlets and roads
    'Waterbody':             2.3,  # Crisp shorelines & rivers; zero boundary bleeding
    'Dry_riverbed':          1.0   # Preserves narrow braided gravel bars
}
#run the entire processing chain
process_tile(lc_probability_path, segments, output_dir,
            river_corridor_path= r"C:\AFLICM_Test\hybrid_lc\mask_layer\TL_River_Corridor_Mask_30m.tif",
            settlement_path = r"C:\AFLICM_Test\hybrid_lc\mask_layer\HRSL_Settlement_Binary_UTM.tif",
            tau=0.65,
            adaptive_sensitivity=0.60,
            smoothness_dict=smoothing_param
            
            )


--- Processing: TL_Prob_2025_mosaic ---
Probabilities: C:\AFLICM_Test\hybrid_lc\Mosaic_LC_Prob_Sentinel\TL_Prob_2025_mosaic.tif
Segments:      C:\AFLICM_Test\hybrid_lc\S2_SNIC_segments\segments_mosaic.tif
Block size:    2048x2048 (overlap: 64px)
Stability tau: 0.65 | Adaptive sensitivity: 0.6
Active Mask-Layer Rules: 2
  1. False Built-up (16) in River Corridor -> Water (17)
  2. False Water ((17, 18)) in Confirmed Settlement -> Built-up (16)
Total windowed blocks: 171 (9 rows x 19 cols)
  Block  10/171 done (  5.8%)...
  Block  30/171 done ( 17.5%)...
  Block  50/171 done ( 29.2%)...
  Block  70/171 done ( 40.9%)...
  Block  90/171 done ( 52.6%)...
  Block 100/171 done ( 58.5%)...
  Block 110/171 done ( 64.3%)...
  Block 120/171 done ( 70.2%)...
  Block 140/171 done ( 81.9%)...

--- Processing Complete! ---
Outputs written to: C:\AFLICM_Test\hybrid_lc\hybrid_outputs\1_new-stage1_assembly\new_test_2025
  1. LULC Map:        C:\AFLICM_Test\hybrid_lc\hybrid_outputs\1_new-stage1_assembly

{'lulc': WindowsPath('C:/AFLICM_Test/hybrid_lc/hybrid_outputs/1_new-stage1_assembly/new_test_2025/TL_Prob_2025_mosaic_LULC.tif'),
 'stability': WindowsPath('C:/AFLICM_Test/hybrid_lc/hybrid_outputs/1_new-stage1_assembly/new_test_2025/TL_Prob_2025_mosaic_Stability_Map.tif'),
 'ci': WindowsPath('C:/AFLICM_Test/hybrid_lc/hybrid_outputs/1_new-stage1_assembly/new_test_2025/TL_Prob_2025_mosaic_Confusion_Index.tif')}

### 2020 Land Cover

In [3]:
#Setup the input and output
lc_probability_2020 = r"C:\AFLICM_Test\hybrid_lc\Mosaic_LC_Prob_Sentinel\TL_Prob_2020_mosaic.tif"
segments = r"C:\AFLICM_Test\hybrid_lc\S2_SNIC_segments\segments_mosaic.tif"
output_dir = Path(r"C:\AFLICM_Test\hybrid_lc\hybrid_outputs\1_new-stage1_assembly\new_test_2020")
#Define custom class weights (Scale: 0.1 to 5.0)
#Higher = trust segment more (smoother); Lower = trust pixel more (sharper)
smoothing_2020 = {
    # Dense contiguous natural vegetation (Strong segment smoothing)
    'Primary_forest':        4.8,
    'Secondary_forest':      4.0,
    'Coastal_forest':        3.5,
    'Mangrove_forest':       3.5,
    
    # Plantations & Agroforestry (Balanced 50/50 object-pixel blend)
    'Teak_plantation':       1.2, #adjusted due to over estimations (original value: 2.5)
    'Eucalyptus_plantation': 1.5,
    'Coffe_agroforestry':    2.5,
    'Cocoa_agroforestry':    2.5,
    'Coconut_agroforestry':  2.0,
    'Mixed_garden':          2.0,
    
    # Open / Herbaceous cover
    'Grassland':             2.5,
    'Shrubland':             2.5,
    'Other_cropland':        2.0,
    'Paddy_field':           1.5,  # Preserves narrow smallholder rice terraces
    
    # Fragmented, dynamic & linear elements (Pixel spectral fidelity dominant)
    'Cleared_land':          1.5,
    'Built-up':              1.5,  # Preserves small hamlets and roads
    'Waterbody':             2.3,  # Crisp shorelines & rivers; zero boundary bleeding
    'Dry_riverbed':          1.0   # Preserves narrow braided gravel bars
}
#run the entire processing chain
process_tile(lc_probability_2020, segments, output_dir,
            river_corridor_path= r"C:\AFLICM_Test\hybrid_lc\mask_layer\TL_River_Corridor_Mask_30m.tif",
            settlement_path = r"C:\AFLICM_Test\hybrid_lc\mask_layer\HRSL_Settlement_Binary_UTM.tif",
            tau=0.65,
            adaptive_sensitivity=0.60,
            smoothness_dict=smoothing_2020
            )


--- Processing: TL_Prob_2020_mosaic ---
Probabilities: C:\AFLICM_Test\hybrid_lc\Mosaic_LC_Prob_Sentinel\TL_Prob_2020_mosaic.tif
Segments:      C:\AFLICM_Test\hybrid_lc\S2_SNIC_segments\segments_mosaic.tif
Block size:    2048x2048 (overlap: 64px)
Stability tau: 0.65 | Adaptive sensitivity: 0.6
Active Mask-Layer Rules: 2
  1. False Built-up (16) in River Corridor -> Water (17)
  2. False Water ((17, 18)) in Confirmed Settlement -> Built-up (16)
Total windowed blocks: 171 (9 rows x 19 cols)
  Block  10/171 done (  5.8%)...
  Block  30/171 done ( 17.5%)...
  Block  50/171 done ( 29.2%)...
  Block  70/171 done ( 40.9%)...
  Block  90/171 done ( 52.6%)...
  Block 100/171 done ( 58.5%)...
  Block 110/171 done ( 64.3%)...
  Block 120/171 done ( 70.2%)...
  Block 140/171 done ( 81.9%)...

--- Processing Complete! ---
Outputs written to: C:\AFLICM_Test\hybrid_lc\hybrid_outputs\1_new-stage1_assembly\new_test_2020
  1. LULC Map:        C:\AFLICM_Test\hybrid_lc\hybrid_outputs\1_new-stage1_assembly

{'lulc': WindowsPath('C:/AFLICM_Test/hybrid_lc/hybrid_outputs/1_new-stage1_assembly/new_test_2020/TL_Prob_2020_mosaic_LULC.tif'),
 'stability': WindowsPath('C:/AFLICM_Test/hybrid_lc/hybrid_outputs/1_new-stage1_assembly/new_test_2020/TL_Prob_2020_mosaic_Stability_Map.tif'),
 'ci': WindowsPath('C:/AFLICM_Test/hybrid_lc/hybrid_outputs/1_new-stage1_assembly/new_test_2020/TL_Prob_2020_mosaic_Confusion_Index.tif')}

## 3. Minimum Mapping Unit Classification Smoothing

In [5]:
from pathlib import Path
from ML_LC_Classifier import apply_mmu

# Paths
lc_2025 = Path(r"C:\AFLICM_Test\hybrid_lc\hybrid_outputs\1_new-stage1_assembly\new_test_2025\TL_Prob_2025_mosaic_LULC.tif")
lc_2020 = Path(r"C:\AFLICM_Test\hybrid_lc\hybrid_outputs\1_new-stage1_assembly\new_test_2020\TL_Prob_2020_mosaic_LULC.tif")

mmu_2020_path = Path(r"C:\AFLICM_Test\hybrid_lc\hybrid_outputs\2_new-stage2_filtering\new_test_lc\intermediate_S2_2020_mosaic_TL_10m.tif")
mmu_2025_path = Path(r"C:\AFLICM_Test\hybrid_lc\hybrid_outputs\2_new-stage2_filtering\new_test_lc\intermediate_S2_2025_mosaic_TL_10m.tif")
#the minimum mapping unit filter table
mmu_table = {
    1: 35,   #primary forest 
    2: 25,   #secondary forest 
    3: 20,   #mangrove forest
    4: 25,   #coastal forest 
    5: 30,   #teak plantation 
    6: 20,   #eucalyptus plantation
    7: 20,   #coffee agroforestry
    8: 25,   #cocoa agroforestry
    9: 20,   #coconut agroforestry
    10: 25,  #mixed garden
    11: 20,  #paddy field
    12: 25,  #other cropland
    13: 20,  #grassland
    14: 30,  #shurbland
    15: 25,  #cleared land
    16: 30,  #built-up
    17: 30,  #Waterbody
}

# 1. Apply MMU to 2020
print("Applying MMU filter to 2020 LULC...")
apply_mmu(
    raster_path=lc_2020,
    output_path=mmu_2020_path,
    table=mmu_table ,
    iterations=4,        # 4 iterations of 3x3 focal fill is plenty for 10-15px patches
    block_size=2048,     # Low-memory streaming window
    overlap=64           # Seam-free boundary overlap
)

# 2. Apply MMU to 2025
print("Applying MMU filter to 2025 LULC...")
apply_mmu(
    raster_path=lc_2025,
    output_path=mmu_2025_path,
    table=mmu_table ,
    iterations=4,
    block_size=2048,
    overlap=64
)
print("MMU filtering complete!")

Applying MMU filter to 2020 LULC...
Applying MMU filter to 2025 LULC...
MMU filtering complete!


# Custom Transition Consistency Analysis
## Define the allowed and forbidden transistion and changing rule

In [10]:
import pandas as pd
import numpy as np
import rasterio
from rasterio.windows import Window
from pathlib import Path

# BACKWARD RULES (T1 <- T2):
# Trust 2025! Correct 2020 to match 2025 ground truth.
# (Also cleans 2020 cloud shadows and seasonal flood boundaries in 2020)
backward_rules = [
    #Any non-primary -> Primary Forest (1)
    (2, 1), (3, 1), (4, 1), (5, 1), (6, 1), (7, 1), (8, 1), (9, 1),
    (10, 1), (11, 1), (12, 1), (13, 1), (14, 1), (15, 1), (16, 1), (17, 1),
    
    #Any non-mangrove -> Mangrove Forest (3)
    (1, 3), (2, 3), (4, 3), (5, 3), (6, 3), (7, 3), (8, 3), (9, 3),
    (10, 3), (11, 3), (12, 3), (13, 3), (14, 3), (16, 3),
    
    # Transitions into Coastal Forest (4)
    (1, 4), (2, 4), (3, 4), (5, 4), (6, 4), (7, 4), (8, 4),
    (10, 4), (11, 4), (12, 4),
    
    # Transitions from Mangrove into terrestrial agriculture
    (3, 2), (3, 5), (3, 6), (3, 7), (3, 8), (3, 9), (3, 10), (3, 12), (3, 13),
    
    # Transitions from Coastal Forest into inland plantations
    (4, 2), (4, 5), (4, 6), (4, 7), (4, 8),
    
    # Cleared Land / Paddy into inland forest
    (11, 2), (11, 4), (15, 2),
    # 2020 Water/Shadows that are verified terrestrial vegetation in 2025:
    # Correct 2020 backward! DO NOT force water into 2025!
    (17, 2), (17, 4), (17, 5), (17, 6), (17, 7), (17, 8), (17, 9),
    (17, 10), (17, 12), (17, 13), (17, 14)
]
# FORWARD RULES (T2 <- T1):
# ONLY Built-up (16) is permanently locked forward 
forward_rules = [
    (16, 2),   # Built-up -> Secondary Forest
    (16, 4),   # Built-up -> Coastal Forest
    (16, 5),   # Built-up -> Teak Plantation
    (16, 6),   # Built-up -> Eucalyptus Plantation
    (16, 7),   # Built-up -> Coffee Agroforestry
    (16, 8),   # Built-up -> Cacao Agroforestry
    (16, 9),   # Built-up -> Coconut Agroforestry
    (16, 10),  # Built-up -> Mixed Garden
    (16, 11),  # Built-up -> Paddy Field
    (16, 12),  # Built-up -> Other Cropland
    (16, 13),  # Built-up -> Grassland
    (16, 14),  # Built-up -> Shrubland
    (16, 17),   # Built-up -> Waterbody
    #Non-teak in 2020 cannot become Teak in 2025 [11 rules]
    #Reverts false 2025 Teak predictions back to true 2020 agricultural/vegetation classes
    (3, 5),   # Mangrove -> Teak
    (4, 5),   # Coastal Forest -> Teak
    (6, 5),   # Eucalyptus Plantation -> Teak
    (7, 5),   # Coffee Agroforestry -> Teak
    (8, 5),   # Cacao Agroforestry -> Teak
    (9, 5),   # Coconut Agroforestry -> Teak
    (10, 5),  # Mixed Garden -> Teak
    (11, 5),  # Paddy Field -> Teak
    (12, 5),  # Other Cropland -> Teak
    (13, 5),  # Grassland -> Teak
    (17, 5)   # Waterbody -> Teak
]

## Run Consistency 

In [11]:
# 2. Memory-Safe Streaming Bidirectional Consistency
t1_2020_mmu = r"C:\AFLICM_Test\hybrid_lc\hybrid_outputs\4_final_consistent\newfinal\TL_2020_LULC_final_rerun.tif"
t2_2025_mmu =r"C:\AFLICM_Test\hybrid_lc\hybrid_outputs\4_final_consistent\newfinal\TL_2025_LULC_final_refined.tif"
out_dir = Path(r"C:\AFLICM_Test\hybrid_lc\hybrid_outputs\4_final_consistent\newfinal")
out_dir.mkdir(parents=True, exist_ok=True)

t1_final = out_dir / "Final_LULC_S2_2020.tif"
t2_final = out_dir / "Final_LULC_S2_2025.tif"
qa_flag  = out_dir / "TL_consistency_QA_flag.tif"

b_set = set(backward_rules)
f_set = set(forward_rules)

with rasterio.open(t1_2020_mmu) as s1, rasterio.open(t2_2025_mmu) as s2:
    meta = s1.profile.copy()
    meta.update(compress="deflate", tiled=True, blockxsize=512, blockysize=512)
    h, w = s1.shape
    block_size = 2048

    with rasterio.open(t1_final, "w", **meta) as dst1, \
         rasterio.open(t2_final, "w", **meta) as dst2, \
         rasterio.open(qa_flag,  "w", **meta) as dst_qa:

        for r in range(0, h, block_size):
            for c in range(0, w, block_size):
                win = Window(c, r, min(block_size, w - c), min(block_size, h - r))
                b1 = s1.read(1, window=win)
                b2 = s2.read(1, window=win)

                out1 = b1.copy()
                out2 = b2.copy()
                qa   = np.zeros_like(b1, dtype=np.uint8)

                # Backward: T1 <- T2
                for (from_c, to_c) in b_set:
                    m = (b1 == from_c) & (b2 == to_c)
                    out1[m] = to_c
                    qa[m] = 1

                # Forward: T2 <- T1
                for (from_c, to_c) in f_set:
                    m = (b1 == from_c) & (b2 == to_c)
                    out2[m] = from_c
                    qa[m] = 2

                dst1.write(out1, 1, window=win)
                dst2.write(out2, 1, window=win)
                dst_qa.write(qa,  1, window=win)

print("Consistent 2020 and 2025 maps successfully generated!")

Consistent 2020 and 2025 maps successfully generated!


## Predefine Consistancy Analysis and Transistion 

In [3]:
from pathlib import Path
from ML_LC_Classifier import (calculate_transition_matrix,apply_transition_consistency,DEFAULT_TIMOR_17_CLASSES)
lc_class = {
    1: "Primary_forest",
    2: "Secondary_forest",
    3: "Mangrove_forest",
    4: "Coastal_forest",
    5: "Teak_plantation",
    6: "Eucalyptus_plantation",
    7: "Coffe_agroforestry",
    8: "Cocoa_agroforestry",
    9: "Coconut_agroforestry",
    10: "Mixed_garden",
    11: "Paddy_field",
    12: "Other_cropland",
    13: "Grassland",
    14: "Shrubland",
    15: "Cleared_land",
    16: "Built-up",
    17: "Waterbody",
}
# 1. Inputs: MMU-filtered 2020 and 2025 rasters
t1_2020 = Path(r"C:\AFLICM_Test\hybrid_lc\hybrid_outputs\4_final_consistent\newfinal\Final_LULC_S2_2020_clean_all.tif")
t2_2025 = Path(r"C:\AFLICM_Test\hybrid_lc\hybrid_outputs\4_final_consistent\newfinal\Final_LULC_S2_2025_clean_all.tif")
# 2. Step 1: Cross-tabulation (Identify existing transitions & area in ha)
print("Computing transition cross-tabulation matrix...")
df_counts, df_areas = calculate_transition_matrix(
    t1_path=t1_2020,
    t2_path=t2_2025,
    class_names=DEFAULT_TIMOR_17_CLASSES,  # Or pass any custom dict: {1: "Forest", ...}
    csv_path="C:/AFLICM_Test/hybrid_lc/hybrid_outputs/transition_matrix_2020_2025_final.csv"
)
# Display top transitions or examine specific pairs
print(df_areas)
# 3. Step 2: Define forbidden (impossible) transitions
# e.g., (from_class, to_class)
# In a 5-year span: Secondary Forest (2) cannot become Primary Forest (1)
# Built-up (16) cannot become Primary Forest (1) or Mangrove (3)
forbidden_rules = [
    (2, 1),   # Secondary Forest -> Primary Forest (impossible in 5 years)
    (13, 1),  # Grassland -> Primary Forest
    (14, 1),  # Shrubland -> Primary Forest
    (15, 1),  # Cleared land -> Primary Forest
    (16, 1),  # Built-up -> Primary Forest
    (16, 2),  # Built-up -> Secondary Forest
    (16, 3),  # Built-up -> Mangrove
]


Computing transition cross-tabulation matrix...
                       Primary_forest  Secondary_forest  Mangrove_forest  \
From (t1)                                                                  
Primary_forest              269657.98            873.12             0.00   
Secondary_forest                 0.00         218152.80             0.00   
Mangrove_forest                  0.00              0.00          2709.40   
Coastal_forest                   0.00              0.00             0.00   
Teak_plantation                  0.00           1346.62             0.00   
Eucalyptus_plantation            0.00            652.53             0.00   
Coffe_agroforestry               0.00           1679.44             0.00   
Cocoa_agroforestry               0.00          16406.35             0.00   
Coconut_agroforestry             0.00            117.92             0.00   
Mixed_garden                     0.00            954.18             0.00   
Paddy_field                      0.00   

4. Step 3: Apply consistency correction to 2025
output_2025_corrected = Path(r"C:\AFLICM_Test\hybrid_lc\hybrid_outputs\3_1_landcover_consistency_tier2\LandCov_S2_2025_mosaic_TL_consistent.tif")
flag_raster = Path(r"C:\AFLICM_Test\hybrid_lc\hybrid_outputs\3_1_landcover_consistency_tier2\TL_transition_corrections_flag.tif")
print("Applying transition consistency...")
apply_transition_consistency(
    t1_path=t1_2020,
    t2_path=t2_2025,
    output_path=output_2025_corrected,
    forbidden=forbidden_rules,
    correction_target="t2",   # Demote/revert impossible 2025 pixels back to realistic classes
    flag_path=flag_raster     # Generates a map of where corrections took place
)
print("Consistency correction completed successfully!")

## Corrected Transition Rule for 2025

In [ ]:
from pathlib import Path
from ML_LC_Classifier import apply_transition_consistency

# Paths
t1_2020 = Path(r"C:\AFLICM_Test\hybrid_lc\hybrid_outputs\MMU_filtered_output\TL_2020_LULC_S2_filtered.tif")
t2_2025 = Path(r"C:\AFLICM_Test\hybrid_lc\hybrid_outputs\MMU_filtered_output\TL_2025_LULC_S2_filtered.tif")

out_2025_corrected = Path(r"C:\AFLICM_Test\hybrid_lc\hybrid_outputs\TL_2025_LULC_Consistent.tif")
flag_raster = Path(r"C:\AFLICM_Test\hybrid_lc\hybrid_outputs\TL_2025_Consistency_Flags.tif")

# 1. Define the complete list of forbidden transitions
forbidden_rules = [
    # Group A: Impossible Primary Forest gains in 5 years
    (2, 1),   # Secondary Forest -> Primary Forest (17,852 ha)
    (5, 1),   # Teak -> Primary Forest
    (6, 1),   # Eucalyptus -> Primary Forest
    (7, 1),   # Coffee -> Primary Forest (15,368 ha)
    (8, 1),   # Cocoa -> Primary Forest (5,944 ha)
    (9, 1),   # Coconut -> Primary Forest
    (10, 1),  # Mixed garden -> Primary Forest
    (11, 1),  # Paddy -> Primary Forest
    (12, 1),  # Other cropland -> Primary Forest
    (13, 1),  # Grassland -> Primary Forest
    (14, 1),  # Shrubland -> Primary Forest
    (15, 1),  # Cleared land -> Primary Forest
    
    # Group B: False Urban De-urbanization
    (16, 1),  # Built-up -> Primary Forest
    (16, 2),  # Built-up -> Secondary Forest
    (16, 3),  # Built-up -> Mangrove
    (16, 17), # Built-up -> Waterbody (urban asphalt/shadow)
    
    # Group C: False Water / Shadow Artifacts
    (1, 17),  # Primary Forest -> Waterbody (mountain shadow)
    (2, 17),  # Secondary Forest -> Waterbody
    (15, 17), # Cleared land -> Waterbody (wet bare soil)
]

# 2. Run consistency correction on 2025
print("Running consistency correction on 2025...")
apply_transition_consistency(
    t1_path=t1_2020,
    t2_path=t2_2025,
    output_path=out_2025_corrected,
    forbidden=forbidden_rules,
    correction_target="t2",   # Corrects 2025 map
    flag_path=flag_raster,     # Maps where corrections were made
    block_size=2048
)

print("2025 Consistency correction complete! Re-calculating final transition matrix...")

In [ ]:
lc_2025 = Path("C:/AFLICM_Test/hybrid_lc/hybrid_outputs/2025 data/TL_Prob_2025_mosaic_LULC.tif")

print("Applying MMU filter to 2025 LULC...")
apply_mmu(
    raster_path=lc_2025,
    output_path=mmu_2025_path,
    table=mmu_table ,
    iterations=5,
    block_size=2048,
    overlap=64
)
print("MMU filtering complete!")

In [ ]:
from ML_LC_Classifier import apply_mmu
lc_2025_pixel = Path(r"C:\AFLICM_Test\hybrid_lc\Pixel_based_Land Cover\LULC_Landsat_2025_pixel.tif")
lc_2020_pixel = Path(r"C:\AFLICM_Test\hybrid_lc\Pixel_based_Land Cover\LULC_Landsat_2020_pixel.tif")
mmu_2020_path = Path(r"C:\AFLICM_Test\hybrid_lc\Pixel_based_Land Cover\LULC_S2_2020_pixel_filtered.tif")
mmu_2025_path = Path(r"C:\AFLICM_Test\hybrid_lc\Pixel_based_Land Cover\LULC_S2_2025_pixel_filtered.tif")
mmu_table = {
    1: 40,   #primary forest 
    2: 30,   #secondary forest 
    3: 20,   #mangrove forest
    4: 30,   #coastal forest 
    5: 15,   #teak plantation 
    6: 20,   #eucalyptus plantation
    7: 20,   #coffee agroforestry
    8: 20,   #cocoa agroforestry
    9: 20,   #coconut agroforestry
    10: 30,  #mixed garden
    11: 30,  #paddy field
    12: 30,  #other cropland
    13: 30,  #grassland
    14: 40,  #shurbland
    15: 30,  #cleared land
    16: 20,  #built-up
    17: 20,  #Waterbody
}
print("Applying MMU filter to 2025 LULC...")

apply_mmu(
    raster_path=lc_2025_pixel,
    output_path=mmu_2025_path,
    table=mmu_table ,
    iterations=5,
    block_size=2048,
    overlap=64
)
print("Applying MMU filter to 2020 LULC...")
apply_mmu(
    raster_path=lc_2020_pixel,
    output_path=mmu_2020_path,
    table=mmu_table ,
    iterations=5,
    block_size=2048,
    overlap=64
)
print("MMU filtering complete!")

In [ ]:
from pathlib import Path
from ML_LC_Classifier import (calculate_transition_matrix,apply_transition_consistency)
lc_class = {
    1: "Primary_forest",
    2: "Secondary_forest",
    3: "Mangrove_forest",
    4: "Coastal_forest",
    5: "Teak_plantation",
    6: "Eucalyptus_plantation",
    7: "Coffe_agroforestry",
    8: "Cocoa_agroforestry",
    9: "Coconut_agroforestry",
    10: "Mixed_garden",
    11: "Paddy_field",
    12: "Other_cropland",
    13: "Grassland",
    14: "Shrubland",
    15: "Cleared_land",
    16: "Built-up",
    17: "Waterbody",
}
# 1. Inputs: MMU-filtered 2020 and 2025 rasters
t1_2020 = Path(r"C:\AFLICM_Test\hybrid_lc\Pixel_based_Land Cover\LULC_S2_2020_pixel_filtered.tif")
t2_2025 = Path(r"C:\AFLICM_Test\hybrid_lc\Pixel_based_Land Cover\LULC_S2_2025_pixel_filtered.tif")
# 2. Step 1: Cross-tabulation (Identify existing transitions & area in ha)
print("Computing transition cross-tabulation matrix...")
df_counts, df_areas = calculate_transition_matrix(
    t1_path=t1_2020,
    t2_path=t2_2025,
    class_names=lc_class,  # Or pass any custom dict: {1: "Forest", ...}
    csv_path="C:/AFLICM_Test/hybrid_lc/hybrid_outputs/transition_matrix_2020_2025_pixel.csv"
)
# Display top transitions or examine specific pairs
print(df_areas)
# 3. Step 2: Define forbidden (impossible) transitions
# e.g., (from_class, to_class)
# In a 5-year span: Secondary Forest (2) cannot become Primary Forest (1)
# Built-up (16) cannot become Primary Forest (1) or Mangrove (3)
forbidden_rules = [
    (2, 1),   # Secondary Forest -> Primary Forest (impossible in 5 years)
    (13, 1),  # Grassland -> Primary Forest
    (14, 1),  # Shrubland -> Primary Forest
    (15, 1),  # Cleared land -> Primary Forest
    (16, 1),  # Built-up -> Primary Forest
    (16, 2),  # Built-up -> Secondary Forest
    (16, 3),  # Built-up -> Mangrove
]

# 4. Step 3: Apply consistency correction to 2025
output_2025_corrected = Path("C:/AFLICM_Test/hybrid_lc/hybrid_outputs/TL_2025_LULC_Consistent_pixel.tif")
flag_raster = Path("C:/AFLICM_Test/hybrid_lc/hybrid_outputs/TL_transition_corrections_flag_pixel.tif")
print("Applying transition consistency...")
apply_transition_consistency(
    t1_path=t1_2020,
    t2_path=t2_2025,
    output_path=output_2025_corrected,
    forbidden=forbidden_rules,
    correction_target="t2",   # Demote/revert impossible 2025 pixels back to realistic classes
    flag_path=flag_raster     # Generates a map of where corrections took place
)
print("Consistency correction completed successfully!")